# 문서 유사도 검색

DB의 BLOB 문서에서 텍스트를 추출하고, 청크별 임베딩을 저장한 뒤 자연어 질의와 가까운 문서 구간을 찾습니다. 이 실습은 검색 결과를 반환하며 LLM 답변 생성은 하지 않습니다.

## 실습 전 준비

DOC_STORE·DOC_STORE_CHUNKS 테이블, 원본 PDF/HWPX 파일, DB에 적재한 MULTILINGUAL_E5_SMALL 모델이 필요합니다. 문서 형식별 텍스트 추출 성공 여부를 먼저 확인합니다.

[공통 환경과 데이터 준비](00.Setup.md)를 먼저 진행합니다. 아래 Python 블록은 같은 세션에서 위에서 아래로 실행하며, 작업 디렉터리는 similarity_search입니다. 코드 블록 자체는 Markdown 화면에서 실행되지 않습니다.

## 처리 흐름

문서 BLOB → 텍스트 추출 → 청크 분할 → 청크 임베딩 → 벡터 저장 → 질의 임베딩 → 거리순 검색

### 0. 파이썬 라이브러리 로딩 및 DB 접속

DB 사용자명과 비밀번호는 매번 입력받고, DB 호스트는 localhost로 고정합니다. 공통 모듈은 user_functions.py를 사용하므로 import_ipynb와 OML4Py는 필요하지 않습니다.


In [ ]:
import os
from getpass import getpass
# Oracle AI Database 접속 및 라이브러리 로딩
import oracledb
import pandas as pd
from user_functions import display, read_lob
from user_functions import get_conn_cursor, get_db_version, exec_sql, dml_sql, ddl_sql, clean_text

# DB 접속
user_id = input("DB 사용자: ").strip()
pwd = getpass("DB 비밀번호: ")
db_service = "orclpdb2.subnet0lab23ai.labswoko.oraclevcn.com"
db_host = "localhost"
dbconn = f"{db_host}:1521/{db_service}"

conn, cursor = get_conn_cursor(user_id,pwd,dbconn)

# DB 버전 확인
display(get_db_version(cursor))


### 1. 샘플 테이블(doc_store) 데이터 조회
문서 저장 테이블 : doc_store
- docno varchar(10) primary key
- doc_dept varchar(40)
- doc_cr_date varchar(20)
- doc_name varchar2(200)
- doc blob

문서번호·부처·제목을 확인해 원본이 적재되었는지 점검합니다. 파일을 Python에서 읽는 것이 아니라 DB의 DOC_STORE에 저장된 BLOB을 처리합니다.


In [ ]:
# 샘플(doc_store) 테이블 데이터 조회

q1 = """ select docno, doc_dept, doc_cr_date, doc_name
         from doc_store
         order by 2, 1 """
exec_sql(cursor,q1)


### 2. 임베딩 모델 및 쿼리 임베딩 확인
- 이 데모에서는 DB에 저장된 임베딩 모델을 사용합니다.

사용자 스키마의 모델을 조회합니다. 모델 이름만 같다고 동일한 모델은 아니므로 적재한 파일과 전처리 설정도 확인합니다.


In [ ]:
# DB관리중인 ONNX 모델 조회
managed_models = """SELECT MODEL_NAME, MINING_FUNCTION,
                    ALGORITHM, ALGORITHM_TYPE, (MODEL_SIZE/1024/1024) MB
                    FROM user_mining_models
                    ORDER BY MODEL_NAME"""

print("DB관리중인 ONNX 모델")
exec_sql(cursor,managed_models)


### 3. 데이터 임베딩 및 인덱싱
- 문서 테이블과 별도로 분리된 테이블(doc_store_chunks)에 문서내용의 청킹 데이터 저장과 벡터 저장을 함.
- 쿼리 임베딩 방법은 오라클 벡터 함수(DBMS_VECTOR_CHAIN)을 사용하여 DB에 저장된 문서에서 텍스트 추출(utl_to_text)->청킹(utl_to_chunks)->임베딩(utl_to_embedding)을 단일 라인 코드로 실행.
- doc_store의 모든 문서의 텍스트를 추출, 워드 100개 단위로 청킹 후, 청크별로 벡터라이징(임베딩)하여 doc_store_chunks 테이블의 벡터 컬럼에 저장.

다음 TRUNCATE는 실습용 청크 테이블을 비웁니다. DDL이므로 롤백할 수 없으며, 재실행할 때만 의도적으로 사용합니다.


In [ ]:
# 기존 데이터 삭제
remove_old_data = "truncate table doc_store_chunks"
ddl_sql(cursor,remove_old_data)


UTL_TO_TEXT → UTL_TO_CHUNKS → UTL_TO_EMBEDDINGS를 연결합니다. 100단어·10% 중첩 설정으로 자른 청크의 ID·내용·벡터를 한 행씩 저장합니다. 추출에 실패하는 형식은 원문을 확인하고 지원 형식으로 변환하거나 적재 대상을 제한합니다.


In [ ]:
# 임베딩 & 벡터 저장
load_vector = """
insert into doc_store_chunks
select dt.docno as docno,et.embed_id as chunk_id, et.embed_data as chunked_data, to_vector(et.embed_vector) embed_vector
from doc_store dt,
   dbms_vector_chain.utl_to_embeddings(
       dbms_vector_chain.utl_to_chunks(dbms_vector_chain.utl_to_text(dt.doc),
       json('{ "by" :
               "words",
               "max" : "100",
               "overlap" : "10",
               "split" : "newline",
               "language" : "korean",
               "normalize": "all"}')),
       json('{"provider":"database",
              "model":"MULTILINGUAL_E5_SMALL"}')) t,
   JSON_TABLE(t.column_value,
              '$' columns (embed_id number,embed_data varchar2(4000),
              embed_vector CLOB)) et
"""
print("벡터라이징 및 저장중입니다........")
dml_sql(conn,cursor,load_vector)


재임베딩하지 않고 저장된 청크를 조회해 텍스트와 벡터가 함께 생성되었는지 확인합니다. 문서번호는 자신의 데이터에 맞게 바꿉니다.


In [ ]:
# 이미 저장한 청크 확인
pd.set_option('display.max_colwidth', 100)
get_vector = """
SELECT docno, chunked_id, chunked_data, embed_vector
FROM doc_store_chunks
WHERE docno = :get_docno
ORDER BY chunked_id
FETCH FIRST 30 ROWS ONLY
"""
result = exec_sql(cursor, get_vector, {"get_docno": "10-3"})
display(result)


HNSW 인덱스는 선택 단계입니다. 관리자에게 VECTOR_MEMORY_SIZE와 메모리 여유를 확인한 뒤 생성합니다. 기존 인덱스를 삭제하면 다시 만들어야 합니다.


In [ ]:
# 벡터 인덱싱
ddl_sql(cursor,"drop index if exists doc_store_chunks_idx")

# 벡터 인덱싱
crIdx = """
create vector index doc_store_chunks_idx on doc_store_chunks(embed_vector)
ORGANIZATION INMEMORY NEIGHBOR GRAPH
WITH DISTANCE COSINE
"""
ddl_sql(cursor,crIdx)

# 벡터 인덱싱 조회
idx_q = """select table_name, index_name, index_type, index_subtype, status
           from user_indexes
           where TABLE_NAME = 'DOC_STORE_CHUNKS'
                 and INDEX_TYPE = 'VECTOR'"""
exec_sql(cursor,idx_q)


### 4. 문서 유사성 검색
- 벡터 공간(doc_store_chunks.ebmed_vector)에 입력문장으로 벡터검색을 실행합니다.

질의에도 문서와 동일한 모델·거리 척도를 적용합니다. FETCH FIRST는 정확 검색의 기준 결과입니다. HNSW 인덱스를 생성했다는 사실만으로 이 SQL이 근사 검색을 사용하는 것은 아닙니다.


In [ ]:
# 검색 문장 입력 및 검색
pd.set_option('display.max_colwidth', 200)
print("예: 고향사랑 기부 상반기 모금 결과")
user_sentence = input("찾으실 문장을 입력하세요:")

sim_q = """
Select T.docno, D.doc_name, T.chunked_id, T.chunked_data AS 문서내용
from doc_store_chunks T, doc_store D
where D.docno = T.docno
order by vector_distance(T.embed_vector,
                         dbms_vector.utl_to_embedding(:u_sentence,
                               json('{"provider":"database",
                                      "model":"MULTILINGUAL_E5_SMALL"}')),
                         cosine)
fetch first 1 rows only
"""
sim_result = exec_sql(cursor,sim_q,{"u_sentence":user_sentence})
sim_result = clean_text(sim_result)
sim_result['문서내용']=sim_result['문서내용'].str.replace('\n', '', regex=False)
display(
    sim_result.head(10).style
    .set_properties(**{'text-align': 'left'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
)


## 연결 정리


In [ ]:
cursor.close()
conn.close()


## 결과 확인

- DOC_STORE의 원본 문서 수와 DOC_STORE_CHUNKS의 청크 수를 구분합니다.
- 질의를 바꾸어 상위 문서·청크가 의도에 맞게 달라지는지 확인합니다. COSINE 거리는 작을수록 가깝습니다.
- 근사 검색을 비교하려면 같은 질의에 FETCH APPROXIMATE FIRST를 사용하고 결과·실행 계획·시간을 함께 확인합니다.

## 참고 문서

- [Oracle 문서 추출·청킹·임베딩](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/convert-file-text-chunks-embeddings-oracle-ai-database.html)
- [Oracle 임베딩 유틸리티](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/utl_to_embedding-and-utl_to_embeddings-dbms_vector.html)

[전체 실습 안내](readme.md) · [공통 함수 설명](user_functions.md)
